B12. Триаж как decision-theoretic задача (Yamin 2026, Cesur 2026).....


In [1]:
import sys, os

def _build_data_tools():
    import json
    import os
    import re
    import warnings
    import numpy as np
    import pandas as pd
    from scipy import stats
    warnings.filterwarnings('ignore')

    def path_join(*parts):
        return os.path.normpath(os.path.join(*parts))

    def normalize_newlines(frame):
        for column in frame.select_dtypes(include=['object', 'string']).columns:
            frame[column] = frame[column].map(lambda value: value.replace('\r\n', '\n').replace('\r', '\n') if isinstance(value, str) else value)
        return frame
    from pathlib import Path
    _start = Path.cwd().resolve()
    _repo = next((p for p in (_start, *_start.parents) if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()), None)
    if _repo is None:
        raise FileNotFoundError('Run the notebook from inside the repository.')
    if os.name == 'nt' and (not str(_repo).startswith(chr(92) * 2 + '?' + chr(92))):
        _repo = Path(chr(92) * 2 + '?' + chr(92) + str(_repo))
    ROOT = str(_repo)
    HERE = str(_repo / 'analyses/clinical_safety')
    OUT = path_join(HERE, 'results')
    os.makedirs(OUT, exist_ok=True)
    PBASE_DIR = 'experiments/clinical_protocols/delphi_panels/heterogeneous_gpt120_organizer_complete'
    HOMOG_DS_DIR = 'experiments/homogeneous_panels/three_deepseekv4_members_complete'
    NB = 'experiments/clinical_protocols/single_model_baselines/baseline_%s/llm_as_judge/baseline_%s_%s_judge.csv'
    USC = 'experiments/token_budget_controls/universal_self_consistency_gpt_seven_samples/llm_as_judge/usc_gpt_sc%d_judge.csv'
    HOMOG_DIRS = {'gpt': 'experiments/homogeneous_panels/three_gpt_members', 'qwen': 'experiments/homogeneous_panels/three_qwen_members', 'gemini': 'experiments/homogeneous_panels/three_gemini_members'}
    PANELS_OTHER_DIR = {'P-weak': 'experiments/clinical_protocols/delphi_panels/panel_qwen_organizer_gemini_deepseek_pro', 'P-mixed': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_gemini_glm', 'P-strong': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_glm_gpt120'}
    PANEL_FILE = {'P-weak': 'delphi_pweak_noroles_judge.csv', 'P-mixed': 'delphi_pmixed_noroles_judge.csv', 'P-strong': 'delphi_pstrong_noroles_judge.csv'}
    SINGLES = ['gpt', 'qwen', 'deepseekv4', 'deepseekv4pro', 'gemini', 'glm', 'kimi']
    COMPOSITION = {'P-base': ['gpt', 'qwen', 'deepseekv4'], 'P-weak': ['qwen', 'gemini', 'deepseekv4pro'], 'P-mixed': ['deepseekv4', 'gemini', 'glm'], 'P-strong': ['deepseekv4', 'glm', 'gpt'], 'homog gpt': ['gpt'] * 3, 'homog qwen': ['qwen'] * 3, 'homog deepseekv4': ['deepseekv4'] * 3, 'homog gemini': ['gemini'] * 3}
    HETERO = {'P-base': 1, 'P-weak': 1, 'P-mixed': 1, 'P-strong': 1, 'homog gpt': 0, 'homog qwen': 0, 'homog deepseekv4': 0, 'homog gemini': 0}
    PANELS = list(COMPOSITION)
    LADDER = ['baseline gpt', 'USC@3 gpt', 'USC@5 gpt', 'USC@7 gpt', 'homog gpt', 'homog deepseekv4', 'P-base']

    def arm_paths(design='noroles'):
        p = {}
        for m in SINGLES:
            p['baseline ' + m] = NB % (m, design, m)
        if design == 'noroles':
            for k in (3, 5, 7):
                p['USC@%d gpt' % k] = USC % k
            for tag, d in HOMOG_DIRS.items():
                p['homog ' + tag] = d + '/llm_as_judge/delphi_homog_%s_noroles_judge.csv' % tag
            p['homog deepseekv4'] = HOMOG_DS_DIR + '/llm_as_judge/delphi_homog_deepseekv4_noroles_judge.csv'
            for k, d in PANELS_OTHER_DIR.items():
                p[k] = d + '/llm_as_judge/' + PANEL_FILE[k]
        p['P-base'] = PBASE_DIR + '/llm_as_judge/delphi_base_%s_judge.csv' % design
        return p
    ROUNDS_DIR = {'P-base': (PBASE_DIR, 'delphi_base_noroles_protocol_r%d_judge.csv'), 'P-weak': (PANELS_OTHER_DIR['P-weak'], 'delphi_pweak_noroles_protocol_r%d_judge.csv'), 'P-mixed': (PANELS_OTHER_DIR['P-mixed'], 'delphi_pmixed_noroles_protocol_r%d_judge.csv'), 'P-strong': (PANELS_OTHER_DIR['P-strong'], 'delphi_pstrong_noroles_protocol_r%d_judge.csv'), 'homog gpt': (HOMOG_DIRS['gpt'], 'delphi_homog_gpt_noroles_protocol_r%d_judge.csv'), 'homog gemini': (HOMOG_DIRS['gemini'], 'delphi_homog_gemini_noroles_protocol_r%d_judge.csv'), 'homog deepseekv4': (HOMOG_DS_DIR, 'delphi_homog_deepseekv4_noroles_protocol_r%d_judge.csv')}
    RETEST = {'P-base': PBASE_DIR + '/llm_as_judge/judge_robustness/delphi_base_%s_retest_r%d_judge.csv', 'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_%s_gpt_retest_r%d_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_%s_qwen_retest_r%d_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_%s_deepseekv4_retest_r%d_judge.csv'}
    SHORT = {'P-base': PBASE_DIR + '/llm_as_judge/judge_robustness/delphi_base_%s_short_judge.csv', 'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_%s_gpt_short_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_%s_qwen_short_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_%s_deepseekv4_short_judge.csv'}
    JUDGE41 = {'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_gpt41/baseline_%s_gpt_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_gpt41/baseline_%s_qwen_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_gpt41/baseline_%s_deepseekv4_judge.csv'}
    CASE_COVARS = ['Type', 'Sub Clinical Specialties', 'Age Group', 'Genealogy', 'Diagnosis10', 'Vignette_race']
    DEXP = 'experiments/panel_parameter_sweeps'
    SWEEPS = {'experts': (DEXP + '/experts/llm_as_judge', 'delphi_experts_%s_judge.csv'), 'questionnaire_items': (DEXP + '/questionnaire_items/llm_as_judge', 'delphi_questionnaire_items_%s_judge.csv'), 'questions_per_expert': (DEXP + '/questions_per_expert/llm_as_judge', 'delphi_questions_per_expert_%s_judge.csv'), 'temperature': (DEXP + '/temperature/llm_as_judge', 'delphi_temperature_%s_judge.csv'), 'rounds': (DEXP + '/rounds/llm_as_judge', 'delphi_%s_judge.csv')}
    EXPERT_GROUPS = {'group1_10models': 10, 'group2_4models': 4, 'group3_5models': 5, 'group4_5models': 5, 'group5_7models': 7, 'group6_5models': 5, 'group7_6models': 6}
    EXT = 'experiments'
    MEDMCQA = {'baseline deepseekv4': EXT + '/external_benchmarks/medmcqa/baseline_deepseekv4/baseline_medmcqa_deepseekv4.csv', 'baseline gpt': EXT + '/external_benchmarks/medmcqa/baseline_gpt/baseline_medmcqa_gpt.csv', 'baseline qwen': EXT + '/external_benchmarks/medmcqa/baseline_qwen/baseline_medmcqa_qwen.csv', 'baseline alice': EXT + '/external_benchmarks/initial_runs/baseline_alice/baseline_medmcqa_alice.csv', 'baseline qwen36': EXT + '/external_benchmarks/initial_runs/baseline_qwen36/baseline_medmcqa_qwen36.csv', 'P-ac (gpt/qwen/ds)': EXT + '/external_benchmarks/medmcqa/panel_gpt120_organizer_qwen_deepseek/delphi_medmcqa_noroles.csv', 'P-ac (gpt/qwen/alice)': EXT + '/external_benchmarks/initial_runs/panel_gpt120_organizer_qwen_alice/delphi_medmcqa_noroles.csv', 'P-deepseektrio': EXT + '/external_benchmarks/initial_runs/three_deepseek_members/delphi_medmcqa_noroles.csv', 'P-base-like': EXT + '/external_benchmarks/initial_runs/heterogeneous_gpt120_organizer/delphi_medmcqa_noroles.csv'}

    def rd_any(rel, index=None):
        p = path_join(ROOT, rel)
        if not os.path.exists(p):
            return None
        with open(p, encoding='utf-8-sig', errors='replace') as fh:
            head = fh.readline()
        sep = ';' if head.count(';') > head.count(',') else ','
        d = normalize_newlines(pd.read_csv(p, sep=sep, encoding='utf-8-sig', low_memory=False))
        if 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        if index and index in d.columns:
            d = d.dropna(subset=[index]).drop_duplicates(index).set_index(index)
        return d

    def sweep_arms(kind, verbose=True):
        import glob as _g
        folder, _ = SWEEPS[kind]
        out = {}
        for p in sorted(_g.glob(path_join(ROOT, folder, '*_judge.csv'))):
            name = os.path.basename(p)[:-len('_judge.csv')]
            d = rd(os.path.relpath(p, ROOT).replace('\\', '/'))
            if d is not None:
                out[name] = d
        if verbose:
            print('свип %s: плеч %d, кейсов %d' % (kind, len(out), len(next(iter(out.values()))) if out else 0))
        return out

    def member_triage_acc(d):
        t = true_label(d)
        out = {}
        for p in participants(d):
            lab = d['r1_emergency__' + p].map(norm_label)
            ok = lab.notna() & t.notna()
            out[p] = float((lab[ok] == t[ok]).mean()) if ok.any() else np.nan
        return pd.Series(out)

    def rd(rel, need_score=True):
        p = path_join(ROOT, rel)
        if not os.path.exists(p):
            return None
        d = normalize_newlines(pd.read_csv(p, sep=';', encoding='utf-8-sig', low_memory=False))
        if 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        if need_score and 'Judge_score' not in d.columns:
            return None
        sub = ['Judge_score', 'Vignette'] if need_score else ['Vignette']
        d = d.dropna(subset=sub).drop_duplicates('Vignette', keep='first')
        return d.set_index('Vignette')

    def load_arms(design='noroles', verbose=True):
        a = {}
        for name, rel in arm_paths(design).items():
            d = rd(rel)
            if d is not None:
                a[name] = d
        if verbose:
            print('плеч загружено: %d' % len(a))
            print('   ' + ', '.join(a))
        return a

    def js(arms, name):
        return pd.to_numeric(arms[name]['Judge_score'], errors='coerce').dropna()

    def score_matrix(arms, names=None):
        names = list(arms) if names is None else names
        m = pd.concat([js(arms, n).rename(n) for n in names], axis=1)
        return m.dropna()

    def meta(arms=None):
        d = rd(PBASE_DIR + '/llm_as_judge/delphi_base_noroles_judge.csv')
        cols = [c for c in CASE_COVARS if c in d.columns]
        return d[cols].copy()

    def item_frame(d):
        r = {}
        if d is None or 'judge_details' not in d.columns:
            return r
        for v, s in d['judge_details'].dropna().items():
            try:
                arr = json.loads(s)
            except Exception:
                continue
            for it in arr:
                ax = next((t.split(':', 1)[1] for t in it.get('tags') or [] if isinstance(t, str) and t.startswith('axis:')), None)
                r[v, it['id']] = {'met': bool(it.get('met')), 'points': it.get('points', 0) or 0, 'axis': ax}
        return r

    def load_items(design='noroles', arms_wanted=None, verbose=True):
        paths = arm_paths(design)
        if arms_wanted:
            paths = {k: v for k, v in paths.items() if k in arms_wanted}
        items = {}
        for name, rel in paths.items():
            d = rd(rel)
            f = item_frame(d)
            if f:
                items[name] = f
        common = None
        for f in items.values():
            common = set(f) if common is None else common & set(f)
        ref = next(iter(items.values()))
        pos = sorted([k for k in common or set() if ref[k]['points'] > 0])
        if verbose:
            print('плеч с попунктными вердиктами: %d' % len(items))
            print('общих положительных пунктов: %d' % len(pos))
        return (items, pos)

    def items_long(items, pos, arms_wanted=None):
        names = arms_wanted or list(items)
        rows = []
        for a in names:
            f = items[a]
            for k in pos:
                it = f[k]
                rows.append({'case': k[0], 'cid': k[1], 'arm': a, 'met': int(it['met']), 'points': it['points'], 'axis': it['axis']})
        return pd.DataFrame(rows)

    def paired(a, b):
        ix = a.index.intersection(b.index)
        x, y = (a.loc[ix].astype(float), b.loc[ix].astype(float))
        d = x - y
        p = stats.wilcoxon(x, y)[1] if (d != 0).any() else np.nan
        t = stats.ttest_rel(x, y)
        return {'n': len(ix), 'delta': float(d.mean()), 'sd': float(d.std(ddof=1)), 'share_gt': float((d > 0).mean()), 'p_wilcoxon': float(p), 't': float(t.statistic), 'p_ttest': float(t.pvalue)}

    def se_clt(x):
        x = np.asarray(x, float)
        n = len(x)
        return float(np.sqrt(x.var(ddof=1) / n))

    def se_clustered(x, cluster, cr1=True):
        x = np.asarray(x, float)
        cl = np.asarray(cluster)
        n = len(x)
        e = x - x.mean()
        df = pd.DataFrame({'e': e, 'c': cl})
        r = df.groupby('c', observed=True)['e'].sum().to_numpy()
        ss = df.groupby('c', observed=True)['e'].apply(lambda s: float((s ** 2).sum())).to_numpy()
        cross = float((r ** 2 - ss).sum())
        v = se_clt(x) ** 2 + cross / n ** 2
        G = len(r)
        if cr1 and G > 1:
            v *= G / (G - 1.0)
        return (float(np.sqrt(max(v, 0.0))), G)

    def cluster_test(d, cluster, cr1=True):
        d = pd.Series(d).astype(float)
        cl = pd.Series(cluster).reindex(d.index)
        ok = cl.notna()
        d, cl = (d[ok], cl[ok])
        m = float(d.mean())
        s0 = se_clt(d)
        s1, G = se_clustered(d, cl, cr1=cr1)
        dfree = max(G - 1, 1)
        t = m / s1 if s1 > 0 else np.nan
        p = float(2 * stats.t.sf(abs(t), dfree)) if np.isfinite(t) else np.nan
        tc = stats.t.ppf(0.975, dfree)
        return {'n': len(d), 'G': G, 'mean': m, 'se_naive': s0, 'se_cluster': s1, 'DEFF': (s1 / s0) ** 2 if s0 > 0 else np.nan, 't': float(t), 'df': dfree, 'p': p, 'ci_lo': m - tc * s1, 'ci_hi': m + tc * s1, 'p_naive': float(2 * stats.norm.sf(abs(m / s0))) if s0 > 0 else np.nan}

    def icc_anova(values, groups):
        df = pd.DataFrame({'y': np.asarray(values, float), 'g': np.asarray(groups)})
        df = df.dropna()
        grp = df.groupby('g', observed=True)['y']
        ni = grp.size().to_numpy().astype(float)
        k = len(ni)
        N = ni.sum()
        if k < 2 or N <= k:
            return (0.0, float(ni.mean() if k else 1), k)
        gm = df['y'].mean()
        msb = float((ni * (grp.mean().to_numpy() - gm) ** 2).sum() / (k - 1))
        ssw = float(((df['y'] - df['g'].map(grp.mean()).to_numpy()) ** 2).sum())
        msw = ssw / (N - k)
        B0 = float((N - (ni ** 2).sum() / N) / (k - 1))
        rho = (msb - msw) / (msb + (B0 - 1) * msw) if msb + (B0 - 1) * msw > 0 else 0.0
        return (float(np.clip(rho, 0, 1)), B0, k)

    def ci_wald(p, n, z=1.96):
        se = np.sqrt(p * (1 - p) / n)
        return (p - z * se, p + z * se)

    def ci_wilson(p, n, z=1.96):
        d = 1 + z ** 2 / n
        c = (p + z ** 2 / (2 * n)) / d
        h = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / d
        return (c - h, c + h)

    def ci_agresti_coull(x, n, z=1.96):
        nt = n + z ** 2
        pt = (x + z ** 2 / 2) / nt
        se = np.sqrt(pt * (1 - pt) / nt)
        return (pt - z * se, pt + z * se)

    def ci_clopper_pearson(x, n, alpha=0.05):
        lo = 0.0 if x == 0 else stats.beta.ppf(alpha / 2, x, n - x + 1)
        hi = 1.0 if x == n else stats.beta.ppf(1 - alpha / 2, x + 1, n - x)
        return (float(lo), float(hi))

    def boot_ci(x, stat=np.mean, B=5000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        x = np.asarray(x, float)
        n = len(x)
        s = np.array([stat(x[rng.integers(0, n, n)]) for _ in range(B)])
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def boot_ci_cluster(x, cluster, stat=np.mean, B=5000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        df = pd.DataFrame({'x': np.asarray(x, float), 'c': np.asarray(cluster)})
        groups = [g['x'].to_numpy() for _, g in df.groupby('c', observed=True)]
        G = len(groups)
        s = []
        for _ in range(B):
            pick = rng.integers(0, G, G)
            s.append(stat(np.concatenate([groups[i] for i in pick])))
        s = np.array(s)
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def holm(pvals):
        p = np.asarray(pvals, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 0.0
        for i, k in enumerate(order):
            run = max(run, (m - i) * p[k])
            adj[k] = min(run, 1.0)
        return adj

    def bh(pvals):
        p = np.asarray(pvals, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 1.0
        for i in range(m - 1, -1, -1):
            k = order[i]
            run = min(run, m * p[k] / (i + 1))
            adj[k] = min(run, 1.0)
        return adj

    def bonferroni(pvals):
        p = np.asarray(pvals, float)
        return np.minimum(p * len(p), 1.0)

    def mcnemar(a, b):
        ix = a.index.intersection(b.index)
        x, y = (a.loc[ix].astype(bool), b.loc[ix].astype(bool))
        n01, n10 = (int((~x & y).sum()), int((x & ~y).sum()))
        if n01 + n10 == 0:
            return {'n01': n01, 'n10': n10, 'p': 1.0}
        return {'n01': n01, 'n10': n10, 'p': float(stats.binomtest(n10, n01 + n10, 0.5).pvalue)}
    ORD = {'SELF': 0, 'NONEMERG': 1, 'EMERG': 2}
    GT = {'Self-care appropriate': 'SELF', 'Requires non-emergent care': 'NONEMERG', 'Requires emergent care': 'EMERG'}
    _DASH = dict.fromkeys(map(ord, '‐‑‒–—―−\xad'), '-')
    _KEYS = [('requires emergent care', 'EMERG'), ('requires non-emergent care', 'NONEMERG'), ('self-care appropriate', 'SELF')]

    def norm_label(x):
        if not isinstance(x, str) or not x.strip():
            return None
        if x.strip() in GT:
            return GT[x.strip()]
        import unicodedata
        s = unicodedata.normalize('NFKC', x).translate(_DASH).lower()
        s = re.sub('\\s+', ' ', re.sub('[^a-z\\- ]', ' ', s)).strip()
        for k, v in _KEYS:
            if s.startswith(k) or k in s:
                return v
        if 'non-emergent' in s or 'non emergent' in s:
            return 'NONEMERG'
        if 'emergent' in s or 'emergency' in s:
            return 'EMERG'
        if 'self' in s:
            return 'SELF'
        return None
    _PAT = re.compile('emergency\\s*type\\s*[:\\-]\\s*[\\*_`#\\s]*([^\\n\\r\\*_|]{0,90})', re.I)

    def parse_triage(text):
        if not isinstance(text, str) or not text.strip():
            return None
        import unicodedata
        m = _PAT.search(unicodedata.normalize('NFKC', text).translate(_DASH))
        if not m:
            return None
        return norm_label('Requires ' + m.group(1) if False else m.group(1))

    def pred_triage(d):
        if 'delphi_consensus_emergency_type' in d.columns:
            s = d['delphi_consensus_emergency_type'].map(norm_label)
            if s.notna().mean() > 0.5:
                return s
        for c in ('winner_protocol', 'baseline_protocol_anonymous'):
            if c in d.columns:
                s = d[c].map(parse_triage)
                if s.notna().mean() > 0.3:
                    return s
        return pd.Series(index=d.index, dtype=object)

    def participants(d):
        return sorted((c.split('__', 1)[1] for c in d.columns if c.startswith('r1_emergency__')))

    def true_label(d):
        return d['Type'].map(GT)

    def words(s):
        return len(re.findall('\\S+', str(s))) if isinstance(s, str) else np.nan

    def fmt_p(p):
        if p is None or (isinstance(p, float) and (not np.isfinite(p))):
            return '  n/a '
        return '%.3f' % p if p >= 0.001 else '%.1e' % p

    def show(df, title=None, floatfmt='%.4f'):
        if title:
            print('\n' + title)
            print('-' * max(len(title), 60))
        with pd.option_context('display.width', 200, 'display.max_columns', 60, 'display.float_format', lambda v: floatfmt % v):
            print(df)

    def save(df, name):
        p = path_join(OUT, name)
        df.to_csv(p, sep=';', encoding='utf-8-sig')
        print('сохранено: out/%s  (%d строк)' % (name, len(df)))
        return p
    return locals()
globals().update({k: v for k, v in _build_data_tools().items() if not k.startswith('_')})
del _build_data_tools
A = load_arms()
ARMS = ['baseline gpt', 'baseline qwen', 'baseline deepseekv4', 'baseline gemini', 'baseline glm', 'baseline deepseekv4pro', 'USC@3 gpt', 'USC@5 gpt', 'USC@7 gpt', 'homog gpt', 'homog qwen', 'homog gemini', 'homog deepseekv4', 'P-weak', 'P-mixed', 'P-strong', 'P-base']


плеч загружено: 18
   baseline gpt, baseline qwen, baseline deepseekv4, baseline deepseekv4pro, baseline gemini, baseline glm, baseline kimi, USC@3 gpt, USC@5 gpt, USC@7 gpt, homog gpt, homog qwen, homog gemini, homog deepseekv4, P-weak, P-mixed, P-strong, P-base


In [2]:
rows = []
PRED = {}
for arm in ARMS:
    if arm not in A:
        continue
    d = A[arm]
    p, t = (pred_triage(d), true_label(d))
    ok = p.notna() & t.notna()
    p, t = (p[ok], t[ok])
    po, to = (p.map(ORD), t.map(ORD))
    PRED[arm] = pd.DataFrame({'pred': po, 'true': to})
    under = po < to
    over = po > to
    sev = (to == 2) & (po == 0)
    miss_em = (to == 2) & (po < 2)
    rows.append({'плечо': arm, 'n': len(p), 'accuracy': float((po == to).mean()), 'недотриаж': float(under.mean()), 'перетриаж': float(over.mean()), 'пропущено EMERG': float(miss_em.mean()), 'EMERG→SELF': int(sev.sum()), 'ср. |ошибка|': float((po - to).abs().mean())})
E = pd.DataFrame(rows).set_index('плечо').sort_values('недотриаж')
show(E, 'Ошибки триажа по плечам')



Ошибки триажа по плечам
------------------------------------------------------------
                          n  accuracy  недотриаж  перетриаж  пропущено EMERG  EMERG→SELF  ср. |ошибка|
плечо                                                                                                 
homog qwen              229    0.7729     0.0000     0.2271           0.0000           0        0.2314
P-weak                  229    0.8079     0.0044     0.1878           0.0044           0        0.1921
homog gemini            229    0.7948     0.0087     0.1965           0.0087           0        0.2052
P-mixed                 229    0.8384     0.0087     0.1528           0.0087           0        0.1616
baseline glm            228    0.8553     0.0088     0.1360           0.0088           0        0.1447
baseline gemini         229    0.8603     0.0131     0.1266           0.0087           0        0.1397
P-strong                229    0.8253     0.0131     0.1616           0.0087           0  

In [3]:
RS = [1, 2, 3, 5, 10, 20, 50, 100]
rows = []
for arm in E.index:
    r = {'плечо': arm}
    for R in RS:
        r['R=%d' % R] = R * E.loc[arm, 'недотриаж'] + E.loc[arm, 'перетриаж']
    rows.append(r)
Cst = pd.DataFrame(rows).set_index('плечо')
show(Cst, 'Ожидаемые издержки (меньше — лучше)')
best = pd.DataFrame({'оптимальное плечо': {'R=%d' % R: Cst['R=%d' % R].idxmin() for R in RS}, 'издержки': {'R=%d' % R: Cst['R=%d' % R].min() for R in RS}})
show(best, 'Кто оптимален при каждом отношении цен')



Ожидаемые издержки (меньше — лучше)
------------------------------------------------------------
                          R=1    R=2    R=3    R=5   R=10   R=20   R=50  R=100
плечо                                                                         
homog qwen             0.2271 0.2271 0.2271 0.2271 0.2271 0.2271 0.2271 0.2271
P-weak                 0.1921 0.1965 0.2009 0.2096 0.2314 0.2751 0.4061 0.6245
homog gemini           0.2052 0.2140 0.2227 0.2402 0.2838 0.3712 0.6332 1.0699
P-mixed                0.1616 0.1703 0.1790 0.1965 0.2402 0.3275 0.5895 1.0262
baseline glm           0.1447 0.1535 0.1623 0.1798 0.2237 0.3114 0.5746 1.0132
baseline gemini        0.1397 0.1528 0.1659 0.1921 0.2576 0.3886 0.7817 1.4367
P-strong               0.1747 0.1878 0.2009 0.2271 0.2926 0.4236 0.8166 1.4716
homog deepseekv4       0.1747 0.1878 0.2009 0.2271 0.2926 0.4236 0.8166 1.4716
P-base                 0.2140 0.2271 0.2402 0.2664 0.3319 0.4629 0.8559 1.5109
baseline deepseekv4pro 0.1184 0.1

In [4]:
def breakeven(a, b):
    ua, oa = (E.loc[a, 'недотриаж'], E.loc[a, 'перетриаж'])
    ub, ob = (E.loc[b, 'недотриаж'], E.loc[b, 'перетриаж'])
    den = ub - ua
    if abs(den) < 1e-12:
        return np.nan
    return (oa - ob) / den
rows = []
for a in ['P-base', 'P-mixed', 'P-strong', 'homog deepseekv4']:
    for b in ['baseline deepseekv4', 'baseline qwen', 'USC@7 gpt', 'baseline gpt']:
        if a not in E.index or b not in E.index:
            continue
        R = breakeven(a, b)
        rows.append({'A': a, 'B': b, 'недотриаж A': E.loc[a, 'недотриаж'], 'недотриаж B': E.loc[b, 'недотриаж'], 'перетриаж A': E.loc[a, 'перетриаж'], 'перетриаж B': E.loc[b, 'перетриаж'], 'R безразличия': R, 'A лучше при': 'R > %.1f' % R if R == R and R > 0 else 'никогда/всегда'})
B = pd.DataFrame(rows)
show(B.set_index(['A', 'B']), 'Отношение цен, при котором A обгоняет B')
print()
print('Читается так: если пропустить неотложный случай хуже перестраховки')
print('более чем в R раз, панель предпочтительнее. Клиническая литература по')
print('неотложной помощи обычно оперирует R в диапазоне 10-50, поэтому')
print('пороги ниже 10 означают «панель предпочтительна практически всегда».')



Отношение цен, при котором A обгоняет B
------------------------------------------------------------
                                      недотриаж A  недотриаж B  перетриаж A  перетриаж B  R безразличия A лучше при
A                B                                                                                                 
P-base           baseline deepseekv4       0.0131       0.0219       0.2009       0.0921        12.3189    R > 12.3
                 baseline qwen             0.0131       0.0218       0.2009       0.0830        13.5000    R > 13.5
                 USC@7 gpt                 0.0131       0.0568       0.2009       0.1397         1.4000     R > 1.4
                 baseline gpt              0.0131       0.0480       0.2009       0.1354         1.8750     R > 1.9
P-mixed          baseline deepseekv4       0.0087       0.0219       0.1528       0.0921         4.6023     R > 4.6
                 baseline qwen             0.0087       0.0218       0.1528       0.08

In [5]:
imp = E.copy()
imp['неявное R (пере/недо)'] = imp['перетриаж'] / imp['недотриаж'].replace(0, np.nan)
imp['осторожность'] = np.where(imp['неявное R (пере/недо)'] > 5, 'консервативная', np.where(imp['неявное R (пере/недо)'] > 2, 'умеренная', 'рискованная'))
show(imp[['accuracy', 'недотриаж', 'перетриаж', 'неявное R (пере/недо)', 'осторожность']].sort_values('неявное R (пере/недо)', ascending=False), 'Неявное отношение цен, заложенное поведением плеча')



Неявное отношение цен, заложенное поведением плеча
------------------------------------------------------------
                        accuracy  недотриаж  перетриаж  неявное R (пере/недо)    осторожность
плечо                                                                                        
P-weak                    0.8079     0.0044     0.1878                43.0000  консервативная
homog gemini              0.7948     0.0087     0.1965                22.5000  консервативная
P-mixed                   0.8384     0.0087     0.1528                17.5000  консервативная
baseline glm              0.8553     0.0088     0.1360                15.5000  консервативная
P-base                    0.7860     0.0131     0.2009                15.3333  консервативная
P-strong                  0.8253     0.0131     0.1616                12.3333  консервативная
homog deepseekv4          0.8253     0.0131     0.1616                12.3333  консервативная
baseline gemini           0.8603     0.01

In [6]:
rows = []
THR = [0.02, 0.05, 0.1, 0.2, 0.3, 0.5]
for arm, D in PRED.items():
    y = (D['true'] == 2).astype(int)
    yhat = (D['pred'] == 2).astype(int)
    n = len(y)
    tp = int(((yhat == 1) & (y == 1)).sum())
    fp = int(((yhat == 1) & (y == 0)).sum())
    r = {'плечо': arm, 'TP': tp, 'FP': fp, 'доля EMERG': float(y.mean())}
    for pt in THR:
        r['NB@%.2f' % pt] = tp / n - fp / n * (pt / (1 - pt))
    rows.append(r)
NB = pd.DataFrame(rows).set_index('плечо')
prev = float(next(iter(PRED.values()))['true'].eq(2).mean())
allpos = {'NB@%.2f' % pt: prev - (1 - prev) * (pt / (1 - pt)) for pt in THR}
NB.loc['[стратегия] всех в EMERG'] = {**{'TP': np.nan, 'FP': np.nan, 'доля EMERG': prev}, **allpos}
NB.loc['[стратегия] никого'] = {**{'TP': 0, 'FP': 0, 'доля EMERG': prev}, **{'NB@%.2f' % pt: 0.0 for pt in THR}}
show(NB.sort_values('NB@0.10', ascending=False), 'Чистая польза (net benefit) при разных порогах')
print()
print('Плечо полезно, только если его NB выше обеих опорных стратегий.')
print('Порог p_t = 0.10 соответствует R = 9, p_t = 0.05 соответствует R = 19.')



Чистая польза (net benefit) при разных порогах
------------------------------------------------------------
                              TP      FP  доля EMERG  NB@0.02  NB@0.05  NB@0.10  NB@0.20  NB@0.30  NB@0.50
плечо                                                                                                     
baseline deepseekv4      82.0000  8.0000      0.3684   0.3589   0.3578   0.3558   0.3509   0.3446   0.3246
P-weak                   83.0000 14.0000      0.3668   0.3612   0.3592   0.3557   0.3472   0.3362   0.3013
baseline deepseekv4pro   82.0000 10.0000      0.3684   0.3588   0.3573   0.3548   0.3487   0.3409   0.3158
baseline gemini          82.0000  8.0000      0.3668   0.3574   0.3562   0.3542   0.3493   0.3431   0.3231
baseline qwen            83.0000 17.0000      0.3668   0.3609   0.3585   0.3542   0.3439   0.3306   0.2882
baseline glm             82.0000 12.0000      0.3684   0.3586   0.3569   0.3538   0.3465   0.3371   0.3070
P-mixed                  82.0000  9

In [7]:
rows = []
for a in ['P-base', 'P-mixed', 'P-strong']:
    for b in ['baseline deepseekv4', 'baseline qwen', 'baseline gpt', 'USC@7 gpt', 'homog deepseekv4']:
        if a not in PRED or b not in PRED:
            continue
        ix = PRED[a].index.intersection(PRED[b].index)
        ca = PRED[a].loc[ix, 'pred'] == PRED[a].loc[ix, 'true']
        cb = PRED[b].loc[ix, 'pred'] == PRED[b].loc[ix, 'true']
        m = mcnemar(ca, cb)
        ua = PRED[a].loc[ix, 'pred'] < PRED[a].loc[ix, 'true']
        ub = PRED[b].loc[ix, 'pred'] < PRED[b].loc[ix, 'true']
        mu = mcnemar(~ua, ~ub)
        rows.append({'A': a, 'B': b, 'acc A': float(ca.mean()), 'acc B': float(cb.mean()), 'p McNemar (acc)': m['p'], 'недотриаж A': float(ua.mean()), 'недотриаж B': float(ub.mean()), 'p McNemar (недотриаж)': mu['p']})
MC = pd.DataFrame(rows)
MC['p acc (Холм)'] = holm(MC['p McNemar (acc)'])
MC['p недо (Холм)'] = holm(MC['p McNemar (недотриаж)'])
show(MC.set_index(['A', 'B']), 'McNemar по точности и по недотриажу, с поправкой Холма')



McNemar по точности и по недотриажу, с поправкой Холма
------------------------------------------------------------
                              acc A  acc B  p McNemar (acc)  недотриаж A  недотриаж B  p McNemar (недотриаж)  p acc (Холм)  p недо (Холм)
A        B                                                                                                                               
P-base   baseline deepseekv4 0.7851 0.8860           0.0001       0.0132       0.0219                 0.6875        0.0018         1.0000
         baseline qwen       0.7860 0.8952           0.0006       0.0131       0.0218                 0.7266        0.0087         1.0000
         baseline gpt        0.7860 0.8166           0.2962       0.0131       0.0480                 0.0215        1.0000         0.2148
         USC@7 gpt           0.7860 0.8035           0.5966       0.0131       0.0568                 0.0020        1.0000         0.0273
         homog deepseekv4    0.7860 0.8253           0.

In [8]:
save(E.reset_index(), 'triage_costs_and_net_benefit_errors.csv')
save(Cst.reset_index(), 'triage_costs_and_net_benefit_costs.csv')
save(B, 'triage_costs_and_net_benefit_breakeven.csv')
save(NB.reset_index(), 'triage_costs_and_net_benefit_net_benefit.csv')
save(MC, 'triage_costs_and_net_benefit_mcnemar.csv')


сохранено: out/triage_costs_and_net_benefit_errors.csv  (17 строк)
сохранено: out/triage_costs_and_net_benefit_costs.csv  (17 строк)
сохранено: out/triage_costs_and_net_benefit_breakeven.csv  (16 строк)
сохранено: out/triage_costs_and_net_benefit_net_benefit.csv  (19 строк)
сохранено: out/triage_costs_and_net_benefit_mcnemar.csv  (15 строк)


'\\\\?\\D:\\research_llmfmed\\research_llmfmed\\1design_alpha\\dataverse_gold\\github_repository\\analyses\\clinical_safety\\results\\triage_costs_and_net_benefit_mcnemar.csv'